## Import Libraries

In [33]:
import os
from openai import OpenAI
import json
import re
import pandas as pd

## Environment Variable

In [ ]:
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## Load Dataset

In [53]:
zomato_data = pd.read_csv(r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\restaurants_enriched.csv")
zomato_data.head()

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,...,food_score,food_mentions,ambiance_score,ambiance_mentions,authenticity_score,authenticity_mentions,service_score,service_mentions,value_for_money_score,value_mentions
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,1,1,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"['Pasta', 'Lunch Buffet', 'Masala Papad', 'Pan...",...,0.723485,20,0.813145,14,0.907675,2,0.776086,12,0.806175,2
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,1,0,4.1,787,080 41714161,Banashankari,Casual Dining,"['Momos', 'Lunch Buffet', 'Chocolate Nirvana',...",...,0.706968,27,0.768720,13,0.747880,5,0.797105,15,0.639608,6
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,1,0,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","['Churros', 'Cannelloni', 'Minestrone Soup', '...",...,0.590025,33,0.573857,16,0.569633,3,0.600343,18,0.507917,3
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,0,0,3.7,88,+91 9620009302,Banashankari,Quick Bites,['Masala Dosa'],...,0.700841,72,0.593326,19,0.816886,13,0.717199,55,0.646975,11
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,0,0,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"['Panipuri', 'Gol Gappe']",...,0.674138,3,0.773325,1,0.500000,0,0.774050,3,0.746950,1


In [54]:
zomato_data.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'listed_in(type)', 'review_ratings',
       'review_texts', 'sentiment_score', 'review_count', 'popularity_score',
       'food_score', 'food_mentions', 'ambiance_score', 'ambiance_mentions',
       'authenticity_score', 'authenticity_mentions', 'service_score',
       'service_mentions', 'value_for_money_score', 'value_mentions'],
      dtype='object')

## OpenAI Query Parser

In [79]:
client = OpenAI()

In [80]:
def parse_user_query(user_query):

    prompt = f"""
You are an NLP engine for a restaurant recommendation system.

Extract the following fields from the user query.

Return ONLY valid JSON.

Fields:
- location
- cuisine
- budget
- online_order
- book_table
- priority

Priority can be one of:
food
ambiance
service
authenticity
value_for_money
popularity
general

If a field is not mentioned return null.

User Query:
{user_query}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": "Return only JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    response_text = response.choices[0].message.content

    # Remove markdown fences if present
    response_text = re.sub(
        r"```json|```",
        "",
        response_text
    ).strip()

    return json.loads(response_text)

## Testing

In [81]:
query = """
Find me an authentic South Indian restaurant
in Jayanagar under ₹800
"""

result = parse_user_query(query)

print(result)

{'location': 'Jayanagar', 'cuisine': 'South Indian', 'budget': 800, 'online_order': None, 'book_table': None, 'priority': 'authenticity'}


## Define Intent Weight Configuration

In [20]:
INTENT_WEIGHTS = {

    "food": {
        "food_score": 0.40,
        "sentiment_score": 0.20,
        "popularity_score": 0.15,
        "service_score": 0.10,
        "ambiance_score": 0.05,
        "authenticity_score": 0.05,
        "value_for_money_score": 0.05
    },

    "authenticity": {
        "authenticity_score": 0.40,
        "food_score": 0.25,
        "sentiment_score": 0.15,
        "service_score": 0.10,
        "popularity_score": 0.10
    },

    "ambiance": {
        "ambiance_score": 0.40,
        "food_score": 0.25,
        "service_score": 0.15,
        "sentiment_score": 0.10,
        "popularity_score": 0.10
    },

    "service": {
        "service_score": 0.40,
        "food_score": 0.20,
        "sentiment_score": 0.15,
        "ambiance_score": 0.10,
        "popularity_score": 0.15
    },

    "value_for_money": {
        "value_for_money_score": 0.40,
        "food_score": 0.20,
        "sentiment_score": 0.15,
        "popularity_score": 0.15,
        "service_score": 0.10
    },

    "general": {
        "food_score": 0.25,
        "sentiment_score": 0.20,
        "ambiance_score": 0.15,
        "service_score": 0.10,
        "authenticity_score": 0.10,
        "value_for_money_score": 0.10,
        "popularity_score": 0.10
    }
}

## Intent Analysis Function

In [42]:
def analyze_intent(parsed_query):

    priority = parsed_query.get(
        "priority",
        "general"
    )

    weights = INTENT_WEIGHTS.get(
        priority,
        INTENT_WEIGHTS["general"]
    )

    return {
        "intent": priority,
        "weights": weights
    }

In [43]:
intent = analyze_intent(
    result
)

print(intent)

{'intent': 'authenticity', 'weights': {'authenticity_score': 0.4, 'food_score': 0.25, 'sentiment_score': 0.15, 'service_score': 0.1, 'popularity_score': 0.1}}


## Merge Both Outputs

This object becomes the input to Candidate Retrieval.

In [44]:
recommendation_request = {
    "filters": result,
    "intent": intent["intent"],
    "weights": intent["weights"]
}
recommendation_request

{'filters': {'location': 'Jayanagar',
  'cuisine': 'South Indian',
  'budget': 800,
  'online_order': None,
  'book_table': None,
  'priority': 'authenticity'},
 'intent': 'authenticity',
 'weights': {'authenticity_score': 0.4,
  'food_score': 0.25,
  'sentiment_score': 0.15,
  'service_score': 0.1,
  'popularity_score': 0.1}}

## Candidate Retrieval Module

In [68]:
def retrieve_candidates(
    restaurant_df: pd.DataFrame,
    filters: dict
) -> pd.DataFrame:

    candidates = restaurant_df.copy()

    # Location Filter
    location = filters.get("location")

    if location:

        candidates = candidates[
            candidates["location"]
            .str.contains(
                location,
                case=False,
                na=False
            )
        ]
    
    # Cuisine Filter
    cuisine = filters.get("cuisine")
    if cuisine:

        cuisine = cuisine.lower()
    
        candidates = candidates[
            candidates["cuisines"]
            .apply(
                lambda cuisines:
                cuisine in str(cuisines).lower()
            )
        ]
    # Budget Filter
    budget = filters.get("budget")

    if budget:
 
        candidates = candidates[
            candidates[
                "approx_cost(for two people)"
            ]
            <= budget
        ]

    # Online Order Filter
    online_order = filters.get(
        "online_order"
    )

    if online_order is not None:

        candidates = candidates[
            candidates["online_order"]
            == int(online_order)
        ]

    # Table Booking Filter
    book_table = filters.get(
        "book_table"
    )

    if book_table is not None:

        candidates = candidates[
            candidates["book_table"]
            == int(book_table)
        ]

    return candidates.reset_index(
        drop=True
    )

In [69]:
candidate_df = retrieve_candidates(
    restaurant_df=zomato_data,
    filters=result
)
candidate_df 

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,...,food_score,food_mentions,ambiance_score,ambiance_mentions,authenticity_score,authenticity_mentions,service_score,service_mentions,value_for_money_score,value_mentions
0,"4, Opposite NMKRV College, 21st C Cross Road, ...",Empire Restaurant,1,0,4.4,4884,080 49653266,Jayanagar,Casual Dining,"['Chicken Tikka Masala', 'Mutton Keema Dosa', ...",...,0.624525,179,0.696569,52,0.612788,17,0.638613,80,0.720622,37
1,"178, Next To Old KEB Office, 8th F Main Road, ...",Biryanis And More,1,0,4.0,618,080 48542442\r\n+91 7337271771,Jayanagar,Casual Dining,"['Prawn Biryani', 'Dragon Chicken', 'Chicken B...",...,0.694362,29,0.611117,3,0.750663,4,0.672312,4,0.688300,2
2,"30th Cross, 8th Main, Near Jain Temple, 4th Bl...",Namma Brahmin's Idli,1,0,3.6,34,+91 8310555879,Jayanagar,Quick Bites,[],...,0.640896,5,0.476575,2,0.832000,1,0.779850,2,0.500000,0
3,"155, 43rd Cross Road, 8th Block, Jayanagar, Ba...",Sri Udupi Food Hub,1,1,4.1,175,+91 9916866033,Jayanagar,Casual Dining,"['Filter Coffee', 'Masala Dosa', 'Idli', 'Vada']",...,0.780858,8,0.677242,5,0.801133,3,0.645363,4,0.742800,3
4,"2nd Floor, Garla Garnet, 9th Main, 4th Block, ...",New Prashanth Hotel,1,0,3.6,143,080 26530150\r\n+91 8861190190,Jayanagar,Casual Dining,"['Ragi Mudde', 'Lemon Chicken', 'Chilli Chicke...",...,0.342300,2,0.908800,1,0.524500,1,0.534450,2,0.908800,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383,"26, 5th Cross, Ayesha Tower, Someshwara Nagar ...",Priya Restaurant,0,0,3.8,0,+91 8970413456\n+91 9036000184,Jayanagar,Quick Bites,[],...,0.500000,0,0.500000,0,0.500000,0,0.500000,0,0.500000,0
384,"28 East End Main Road, Next To Pump House, Jay...",Jayanagara Donne Biryani,1,0,3.6,62,+91 8050935759,Jayanagar,Quick Bites,[],...,0.554891,54,0.321238,7,0.486950,7,0.496456,13,0.458225,11
385,"401, Swagath Main Road, Tilak Nagar, Jayanagar...",New Select Hotel,0,0,3.4,0,+91 9739001272,Jayanagar,Quick Bites,[],...,0.500000,0,0.500000,0,0.500000,0,0.500000,0,0.500000,0
386,"2, Ashoka Pillar Circle, 2nd Block, Jayanagar,...",Samskruti - Sanman Gardenia,1,0,4.0,141,080 26570711\n+91 9663684499,Jayanagar,Casual Dining,"['Roti', 'Butter Kulcha', 'Fried Rice', 'Tomat...",...,0.733283,6,0.500000,0,0.500000,0,0.719875,4,0.500000,0


## Ranking Function

In [70]:
def rank_restaurants(
    candidate_df,
    weights
):

    candidate_df = candidate_df.copy()

    candidate_df["final_score"] = 0

    for feature, weight in weights.items():

        if feature in candidate_df.columns:

            candidate_df["final_score"] += (
                candidate_df[feature] * weight
            )

    return (
        candidate_df
        .sort_values(
            "final_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

In [73]:
ranked_df = rank_restaurants(
    candidate_df=candidate_df,
    weights=recommendation_request["weights"]
)


In [74]:
ranked_df[
    [
        "name",
        "final_score"
    ]
].head(10)

,name,final_score
0,Shree Venkateshwara North Karnataka Hotel,0.895243
1,Vasudev Adiga's,0.843407
2,Vasudev Adiga's,0.843377
3,Vybhava,0.823324
4,Vybhava,0.823324
5,Vybhava,0.823324
6,Sri Sai 99 Variety Dosa,0.808183
7,Sri Sai 99 Variety Dosa,0.808183
8,Sri Sai 99 Variety Dosa,0.808183
9,Sri Sai 99 Variety Dosa,0.808183


## Top-N Recommendation

In [75]:
def get_top_n(
    ranked_df,
    n=10
):

    return ranked_df.head(n)

In [77]:
top_restaurants = get_top_n(
    ranked_df,
    n=10
)
top_restaurants

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,...,food_mentions,ambiance_score,ambiance_mentions,authenticity_score,authenticity_mentions,service_score,service_mentions,value_for_money_score,value_mentions,final_score
0,"1/21, Vijaylaxmi Towers, 32 Cross, 10th Main, ...",Shree Venkateshwara North Karnataka Hotel,0,0,4.0,82,+91 9945612307,Jayanagar,Quick Bites,"['Buttermilk', 'Thali', 'Brinjal Curry', 'Sala...",...,4,0.582275,2,0.9593,1,0.968000,1,0.811925,3,0.895243
1,"36, 12th Main, 27th Cross , 4th Block, Jayanag...",Vasudev Adiga's,0,0,3.8,266,080 22940240\r\n080 22940235,Jayanagar,Quick Bites,"['Puri Saagu', 'Idli Sambar', 'Filter Coffee',...",...,2,0.500000,0,0.9361,1,0.500000,0,0.500000,0,0.843407
2,"36, 12th Main, 27th Cross , 4th Block, Jayanag...",Vasudev Adiga's,0,0,3.8,265,080 22940240\r\n080 22940235,Jayanagar,Quick Bites,"['Puri Saagu', 'Idli Sambar', 'Filter Coffee',...",...,2,0.500000,0,0.9361,1,0.500000,0,0.500000,0,0.843377
3,"35-1, Near Yediyur Lake, Kanakapura Road, Jaya...",Vybhava,0,0,3.8,39,+91 8026653198,Jayanagar,Quick Bites,"['Masala Dosa', 'Idli', 'Vada Sambar']",...,2,0.500000,0,0.8454,1,0.796175,2,0.796175,2,0.823324
4,"35-1, Near Yediyur Lake, Kanakapura Road, Jaya...",Vybhava,0,0,3.8,39,+91 8026653198,Jayanagar,Quick Bites,"['Masala Dosa', 'Idli', 'Vada Sambar']",...,2,0.500000,0,0.8454,1,0.796175,2,0.796175,2,0.823324
5,"35-1, Near Yediyur Lake, Kanakapura Road, Jaya...",Vybhava,0,0,3.8,39,+91 8026653198,Jayanagar,Quick Bites,"['Masala Dosa', 'Idli', 'Vada Sambar']",...,2,0.500000,0,0.8454,1,0.796175,2,0.796175,2,0.823324
6,"Big Bazar Compound, Opposite Central Mall, 9th...",Sri Sai 99 Variety Dosa,1,0,3.6,38,+91 7899689049,Jayanagar,Quick Bites,[],...,1,0.500000,0,0.8893,1,0.720200,1,0.500000,0,0.808183
7,"Big Bazar Compound, Opposite Central Mall, 9th...",Sri Sai 99 Variety Dosa,1,0,3.6,38,+91 7899689049,Jayanagar,Quick Bites,[],...,1,0.500000,0,0.8893,1,0.720200,1,0.500000,0,0.808183
8,"Big Bazar Compound, Opposite Central Mall, 9th...",Sri Sai 99 Variety Dosa,1,0,3.6,38,+91 7899689049,Jayanagar,Quick Bites,[],...,1,0.500000,0,0.8893,1,0.720200,1,0.500000,0,0.808183
9,"Big Bazar Compound, Opposite Central Mall, 9th...",Sri Sai 99 Variety Dosa,1,0,3.6,38,+91 7899689049,Jayanagar,Quick Bites,[],...,1,0.500000,0,0.8893,1,0.720200,1,0.500000,0,0.808183
